In [ ]:
### INITIALIZE VARIABLES ###
project_name = "ee-lequangminh03" # Name of the project on your google cloud project
drive_folder= "New Satellite Images ( Full 13 bands )" # Name of the folder in your google drive where the data is stored
width=0.05
height=0.05


import ee
# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize(project=project_name)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
import matplotlib.patches as patches
import ee

def parse_multipolygon(multipolygon_wkt):
    """Parse WKT MultiPolygon to Shapely MultiPolygon."""
    import re
    coords_str = re.findall(r'\(([^()]+)\)', multipolygon_wkt)[-1]
    coords = [list(map(float, pair.split())) for pair in coords_str.split(',')]
    return MultiPolygon([Polygon(coords)])

def slice_multipolygon(multipolygon, grid_size=0.05):
    """
    Slice a MultiPolygon into grid rectangles of specified size.

    Args:
        multipolygon (MultiPolygon): Input multipolygon
        grid_size (float): Size of grid rectangles in degrees

    Returns:
        list: List of grid rectangles that intersect the multipolygon
    """
    # Get bounds of the multipolygon
    minx, miny, maxx, maxy = multipolygon.bounds

    # Adjust bounds to ensure full coverage
    minx = np.floor(minx / grid_size) * grid_size
    miny = np.floor(miny / grid_size) * grid_size
    maxx = np.ceil(maxx / grid_size) * grid_size
    maxy = np.ceil(maxy / grid_size) * grid_size

    # Generate grid
    grid_rectangles = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            # Create rectangle
            rect = Polygon([
                (x, y),
                (x + grid_size, y),
                (x + grid_size, y + grid_size),
                (x, y + grid_size)
            ])

            # Check intersection
            intersection = rect.intersection(multipolygon)
            if not intersection.is_empty:
                grid_rectangles.append({
                    'rectangle': ee.Geometry.Rectangle([x, y, x + grid_size, y + grid_size]),
                    'position': "None"
                })

            y += grid_size
        x += grid_size

    return grid_rectangles

def visualize_grid_rectangles(multipolygon, grid_rectangles):
    """
    Visualize the multipolygon and grid rectangles

    Args:
        multipolygon (MultiPolygon): Original multipolygon
        grid_rectangles (list): List of grid rectangles
    """
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(15, 10))

    # Plot original multipolygon
    minx, miny, maxx, maxy = multipolygon.bounds

    # Plot grid rectangles
    for rect in grid_rectangles:
        # Create rectangle patch
        bbox = rect['bbox']
        intersection_ratio = rect['area_intersection']

        # Color intensity based on intersection ratio
        color = plt.cm.Blues(intersection_ratio)

        # Add rectangle patch
        rect_patch = patches.Rectangle(
            (bbox[0], bbox[1]),
            bbox[2] - bbox[0],
            bbox[3] - bbox[1],
            fill=True,
            color=color,
            alpha=0.6,
            edgecolor='black',
            linewidth=0.5
        )
        ax.add_patch(rect_patch)

    # Set plot limits
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    # Labels and title
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'Grid Rectangles (Total: {len(grid_rectangles)})')

    # Add colorbar to show intersection ratio
    sm = plt.cm.ScalarMappable(cmap=plt.cm.Blues)
    sm.set_array([])  # You have to do this step for this colorbar to work
    plt.colorbar(sm, label='Intersection Ratio')

    # Display grid
    ax.grid(True, linestyle='--', linewidth=0.5)

    plt.tight_layout()
    plt.show()


In [ ]:
import ee
# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize(project=project_name)

# Define the polygon around Canberra, Australia large enough for 8 rectangles

# hoang_hoa_poly = [
#     [105.7595, 19.9614],
#     [105.9666, 19.9614],
#     [105.9666, 19.7699],
#     [105.7595, 19.7699],
#     [105.7595, 19.9614]
# ]
# quan_son_poly=[
#     [104.6063, 20.4116],
#     [105.1309, 20.4116],
#     [105.1309, 20.0953],
#     [104.6063, 20.0953],
#     [104.6063, 20.4116]
# ]

# hoang_hoa_polygon = ee.Geometry.Polygon(hoang_hoa_poly)
# quan_son_polygon = ee.Geometry.Polygon(quan_son_poly)
# Parameters for the number of rectangles

def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
             .And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

# # Function to subdivide the polygon into rectangles based on the number of rectangles
# def subdivide_polygon(polygon, numRectsX, numRectsY):
#     bounds = polygon.bounds()
#     coords = bounds.coordinates().get(0).getInfo()
#     xMin = coords[0][0]
#     xMax = coords[2][0]
#     yMin = coords[0][1]
#     yMax = coords[2][1]

#     rectWidth = (xMax - xMin) / numRectsX
#     rectHeight = (yMax - yMin) / numRectsY

#     rectangles = []

#     for x in range(numRectsX):
#         for y in range(numRectsY):
#             x0 = xMin + x * rectWidth
#             y0 = yMin + y * rectHeight
#             x1 = x0 + rectWidth
#             y1 = y0 + rectHeight
#             rect = {
#                 "rectangle":ee.Geometry.Rectangle([x0, y0, x1, y1]),
#                 "position": (x, y)
#             }
#             rectangles.append(rect)

#     return rectangles

# Function to export images to Google Drive
def prepare_export_tasks(place_name: str, rectangles, from_date: str, to_date: str ):
    tasks = []
    for index, rectangle in enumerate(rectangles):
        rect=rectangle['rectangle']
        position=rectangle['position']
        image = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
            .filterBounds(rect) \
            .filterDate(from_date, to_date) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \

        image=image.map(maskS2clouds)\
            .median() \
            .select(['B1','B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']) \
            .clip(rect)

        task = ee.batch.Export.image.toDrive(
            image=image,
            description = f"{place_name}_2024_{str(position[0])}_{str(position[1])}_{str(index)}",
            folder=drive_folder,
            scale=5,  # Adjust this scale as needed
            region=rect,
            maxPixels=1e9,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized': True}
        )
        tasks.append(task)

    return tasks

# Call the function to prepare the tasks
multipolygon_wkt = """
    MULTIPOLYGON (((105.78835682561 21.3687531851594,105.759025263863 21.4190358624393,105.666840355517 21.4776989859324,105.553704331637 21.5908350098121,105.478280315718 21.5615034480655,105.390285630478 21.532171886319,105.293910499025 21.5363621094256,105.281339829705 21.4693185397191,105.348383399411 21.385514077586,105.377714961158 21.4022749700126,105.415426969118 21.3142802847729,105.369334514945 21.3268509540928,105.314861614558 21.2891389461329,105.327432283878 21.2220953764265,105.256198491065 21.1927638146799,105.256198491065 21.0921984601201,105.302290945238 21.008393997987,105.423807415331 20.9874428824538,105.469899869504 20.9832526593471,105.461519423291 20.912018866534,105.457329200184 20.8449752968275,105.511802100571 20.7988828426543,105.545323885424 20.7569806115877,105.574655447171 20.7150783805212,105.612367455131 20.6815565956679,105.645889239984 20.6773663725613,105.666840355517 20.6312739183881,105.679411024837 20.5893716873215,105.733883925224 20.5851814642149,105.759025263863 20.5600401255749,105.792547048717 20.522328117615,105.85121017221 20.6270836952814,105.859590618423 20.6522250339213,105.922443965023 20.6606054801346,105.981107088516 20.6731761494546,106.006248427156 20.6983174880945,106.035579988903 20.656415257028,106.052340881329 20.5893716873215,106.131955120356 20.6061325797481,106.152906235889 20.6396543646014,106.152906235889 20.656415257028,106.190618243849 20.6522250339213,106.219949805596 20.6522250339213,106.270232482875 20.6899370418812,106.286993375302 20.7025077112012,106.374988060542 20.7108881574145,106.387558729862 20.7150783805212,106.421080514715 20.6354641414947,106.467172968888 20.5684205717882,106.550977431021 20.6019423566415,106.638972116261 20.6187032490681,106.731157024607 20.6312739183881,106.764678809461 20.6773663725613,106.831722379167 20.7025077112012,106.802390817421 20.7695512809077,106.798200594314 20.7988828426543,106.835912602274 20.8282144044009,106.87781483334 20.7569806115877,106.944858403047 20.7695512809077,106.982570411007 20.7234588267345,107.032853088287 20.6773663725613,107.125037996633 20.7108881574145,107.125037996633 20.7486001653744,107.21722290498 20.7360294960544,107.242364243619 20.777931727121,107.317788259539 20.8072632888676,107.351310044392 20.7569806115877,107.477016737592 20.668985926348,107.535679861085 20.7737415040143,107.523109191765 20.8156437350809,107.565011422832 20.9078286434273,107.590152761472 20.9413504282806,107.602723430792 20.9874428824538,107.640435438752 21.0712473445869,107.690718116032 21.1341006911867,107.757761685738 21.2011442608932,107.845756370978 21.263997607493,107.921180386898 21.3100900616662,107.958892394858 21.3310411771995,108.042696856991 21.3897043006927,108.080408864951 21.4609380935058,108.017555518351 21.6159763484521,107.988223956604 21.6243567946654,108.000794625924 21.5698838942789,107.942131502431 21.6075959022388,107.916990163791 21.6327372408787,107.883468378938 21.6578785795186,107.837375924765 21.6788296950519,107.761951908845 21.6662590257319,107.707479008458 21.6494981333053,107.673957223605 21.628547017772,107.640435438752 21.6201665715587,107.577582092152 21.6159763484521,107.548250530405 21.6159763484521,107.531489637979 21.6159763484521,107.506348299339 21.641117687092,107.485397183805 21.6578785795186,107.443494952739 21.6830199181585,107.389022052352 21.6243567946654,107.334549151966 21.6034056791321,107.225603351193 21.5950252329188,107.196271789446 21.5740741173855,107.19208156634 21.532171886319,107.200462012553 21.4818892090391,107.208842458766 21.4651283166125,107.21722290498 21.4441772010792,107.225603351193 21.4106554162259,107.200462012553 21.3897043006927,107.171130450806 21.3897043006927,107.133418442846 21.385514077586,107.095706434887 21.385514077586,107.070565096247 21.3771336313727,107.037043311393 21.3603727389461,107.020282418967 21.3436118465195,106.99095085722 21.3226607309862,106.96580951858 21.2975193923463,106.95323884926 21.276568276813,106.944858403047 21.2472367150664,106.944858403047 21.2179051533198,106.9155268413 21.2011442608932,106.898765948874 21.1969540377865,106.848483271594 21.1969540377865,106.764678809461 21.1927638146799,106.764678809461 21.1718126991466,106.764678809461 21.1718126991466,106.710205909074 21.1760029222533,106.638972116261 21.1843833684666,106.571928546555 21.2053344839998,106.538406761701 21.2220953764265,106.530026315488 21.2346660457464,106.500694753741 21.2430464919597,106.488124084421 21.2430464919597,106.442031630248 21.2430464919597,106.383368506755 21.2388562688531,106.349846721902 21.2220953764265,106.337276052582 21.2053344839998,106.328895606369 21.1885735915732,106.316324937049 21.1718126991466,106.295373821515 21.1382909142934,106.270232482875 21.1508615836133,106.266042259769 21.1676224760399,106.215759582489 21.1843833684666,106.198998690062 21.1843833684666,106.178047574529 21.1969540377865,106.173857351422 21.2137149302132,106.161286682102 21.2179051533198,106.123574674142 21.2262855995331,106.090052889289 21.2262855995331,106.048150658223 21.251426938173,106.039770212009 21.2556171612797,106.035579988903 21.2598073843864,105.981107088516 21.276568276813,105.93920485745 21.2681878305997,105.918253741916 21.3058998385596,105.897302626383 21.3519922927328,105.87635151085 21.3687531851594,105.86378084153 21.3813238544794,105.855400395317 21.3938945237993,105.821878610463 21.398084746906,105.779976379397 21.3938945237993,105.77578615629 21.3938945237993,105.78835682561 21.3687531851594)))
    """
    # Parse the multipolygon
mpoly = parse_multipolygon(multipolygon_wkt)

    # Slice into grid
grid_rectangles = slice_multipolygon(mpoly)



tasks=[]

quanson_temp = prepare_export_tasks('Area', grid_rectangles, f'2024-01-01', f'2024-12-01')

tasks.extend(quanson_temp)

# tasks=prepare_export_tasks('HoangHoa', [hoang_hoa_rectangles[0]], '2023-03-01', '2023-04-01')
# tasks=prepare_export_tasks('QuanSon', quan_son_rectangles, '2023-09-01', '2023-10-01')
# Start the tasks
for task in tasks:
    task.start()
    print(f"Exporting {task.config['description']}")

Exporting Area_2024_N_o_0
Exporting Area_2024_N_o_1
Exporting Area_2024_N_o_2
Exporting Area_2024_N_o_3
Exporting Area_2024_N_o_4
Exporting Area_2024_N_o_5
Exporting Area_2024_N_o_6
Exporting Area_2024_N_o_7
Exporting Area_2024_N_o_8
Exporting Area_2024_N_o_9
Exporting Area_2024_N_o_10
Exporting Area_2024_N_o_11
Exporting Area_2024_N_o_12
Exporting Area_2024_N_o_13
Exporting Area_2024_N_o_14
Exporting Area_2024_N_o_15
Exporting Area_2024_N_o_16
Exporting Area_2024_N_o_17
Exporting Area_2024_N_o_18
Exporting Area_2024_N_o_19
Exporting Area_2024_N_o_20
Exporting Area_2024_N_o_21
Exporting Area_2024_N_o_22
Exporting Area_2024_N_o_23
Exporting Area_2024_N_o_24
Exporting Area_2024_N_o_25
Exporting Area_2024_N_o_26
Exporting Area_2024_N_o_27
Exporting Area_2024_N_o_28
Exporting Area_2024_N_o_29
Exporting Area_2024_N_o_30
Exporting Area_2024_N_o_31
Exporting Area_2024_N_o_32
Exporting Area_2024_N_o_33
Exporting Area_2024_N_o_34
Exporting Area_2024_N_o_35
Exporting Area_2024_N_o_36
Exporting A